# XGBoost Ablation: Remove Worst Concentration + Gene Hotspots

This notebook removes the worst meaningful `Concentration + Gene` hotspots from the XGBoost by-gene investigation, then retrains and re-analyzes the model with the same frozen by-gene+mRNA setup.

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.merge_historic_data import load_merged_dataset
from utils.pipeline import SiRNADataPipeline
from utils.splitter import GroupKFoldLeakPerGroup

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Build Current Pipeline Data

This ablation uses the current shared preprocessing setup: `strict_cleaning=True`, `add_mrna=True`, and `use_normalized_conditions=False`. It also keeps the frozen XGBoost by-gene+mRNA Optuna hyperparameters from the earlier baseline notebook so that the only change is the row removal.

In [3]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

raw_df = load_merged_dataset(cmsirna_path, historic_path)
pipeline = SiRNADataPipeline(target_len=25, fetch_missing_mrna=True)
enriched_df = pipeline.enrich_dataset_with_encodings(
    raw_df,
    strict_cleaning=True,
    add_mrna=True,
)

enriched_df["patent_group"] = enriched_df.get(
    "patent_ID", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")
enriched_df["Authorization_status"] = enriched_df.get(
    "Authorization_status", pd.Series(index=enriched_df.index, dtype=object)
).fillna("UNKNOWN")
enriched_df["Title"] = enriched_df.get(
    "Title", pd.Series(index=enriched_df.index, dtype=object)
).fillna("HISTORIC_OR_UNKNOWN")


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

## What This Notebook Removes

Removed hotspots:
- `APP @ 20 nM`
- `INHBE @ 100 nM`
- `PCSK9 @ 100 nM`

Why: these are among the worst meaningful `Concentration + Gene` hotspots in the by-gene XGBoost notebook. I intentionally prioritized meaningful sample sizes over tiny extreme groups, so this notebook tests whether removing high-impact target-specific transfer failures improves the overall signal.

Exact rows from the investigation table:
- `APP @ 20 nM`: `n_samples = 168`, `spearman = -0.556349`, `mae = 38.583055`, `mean_true = 41.410714`, `mean_pred = 57.806583`
- `INHBE @ 100 nM`: `n_samples = 180`, `spearman = -0.440447`, `mae = 37.574473`, `mean_true = 62.262167`, `mean_pred = 39.903522`
- `PCSK9 @ 100 nM`: `n_samples = 293`, `spearman = -0.212300`, `mae = 31.238838`, `mean_true = 54.427304`, `mean_pred = 54.176559`

These are the worst larger-sample gene hotspots, so they are better ablation candidates than tiny unstable groups such as 13-row or 16-row slices.


In [4]:
combo_mask = (
    ((enriched_df["gene_target_symbol_name"] == "APP") & (np.isclose(enriched_df["Concentration_nM"], 20.0, equal_nan=False)))
    | ((enriched_df["gene_target_symbol_name"] == "INHBE") & (np.isclose(enriched_df["Concentration_nM"], 100.0, equal_nan=False)))
    | ((enriched_df["gene_target_symbol_name"] == "PCSK9") & (np.isclose(enriched_df["Concentration_nM"], 100.0, equal_nan=False)))
)
removed_df = enriched_df.loc[combo_mask].copy()
filtered_df = enriched_df.loc[~combo_mask].reset_index(drop=True)

removal_summary = pd.DataFrame([{
    "starting_rows": int(len(enriched_df)),
    "rows_removed": int(len(removed_df)),
    "rows_remaining": int(len(filtered_df)),
    "pct_removed": float(len(removed_df) / len(enriched_df)),
    "removed_hotspots": "APP@20, INHBE@100, PCSK9@100",
}])
removal_summary

X, groups, y = pipeline.prepare_for_classical_ml(
    filtered_df,
    target_column="Inhibition",
    use_normalized_conditions=False,
)
mask = ~np.isnan(y)
X = X[mask]
groups = groups[mask]
y = y[mask]
analysis_df = filtered_df.loc[mask].reset_index(drop=True).copy()

print("Filtered dataframe shape:", analysis_df.shape)
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Unique genes:", len(np.unique(groups)))


Feature matrix X shape: (34799, 1425), target y shape: (34799,)
Filtered dataframe shape: (34799, 40)
Feature matrix shape: (34799, 1425)
Target shape: (34799,)
Unique genes: 54


## Helper Functions

In [5]:
def make_model(frozen_params):
    return XGBRegressor(
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
        **frozen_params,
    )


def evaluate(y_true, y_pred):
    return {
        "spearman": float(spearmanr(y_true, y_pred).statistic),
        "pearson": float(pearsonr(y_true, y_pred).statistic),
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
    }


def grouped_spearman(df, group_cols, min_samples=10):
    rows = []
    for keys, group_df in df.groupby(group_cols, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        if pd.isna(corr):
            continue
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: value for col, value in zip(group_cols, keys)}
        row.update({
            "n_samples": len(group_df),
            "spearman": float(corr),
            "mae": float(group_df["abs_error"].mean()),
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


def issue_summary(df, group_col, min_samples=20):
    rows = []
    for value, group_df in df.groupby(group_col, dropna=False):
        if len(group_df) < min_samples:
            continue
        corr = spearmanr(group_df["y_true"], group_df["y_pred"], nan_policy="omit").statistic
        rows.append({
            group_col: value,
            "n_samples": len(group_df),
            "mae": float(group_df["abs_error"].mean()),
            "rmse": float(np.sqrt(np.mean(np.square(group_df["residual"])))),
            "spearman": float(corr) if not pd.isna(corr) else np.nan,
            "mean_true": float(group_df["y_true"].mean()),
            "mean_pred": float(group_df["y_pred"].mean()),
        })
    return pd.DataFrame(rows).sort_values(["spearman", "mae"], ascending=[True, False]).reset_index(drop=True)


## Refit And Re-Analyze

The model is retrained from scratch after the hotspot removal, then evaluated again with the same by-gene out-of-fold procedure used in the main XGBoost investigation notebook.

In [6]:
frozen_params = {'n_estimators': 800, 'max_depth': 4, 'learning_rate': 0.15881823130907038, 'subsample': 0.8812898741586134, 'colsample_bytree': 0.7824379872752019, 'min_child_weight': 4, 'reg_lambda': 0.8342807691178866, 'reg_alpha': 1.4296995092035882, 'gamma': 0.07531958697602548}
baseline_metrics = pd.DataFrame([{'spearman': 0.45886, 'pearson': 0.453422, 'rmse': 31.421689, 'mae': 24.568816}])
gene_cv = GroupKFoldLeakPerGroup(n_splits=3, leak_n=30, random_state=42)

fold_rows = []
oof_frames = []
oof_true = []
oof_pred = []

for fold_id, (train_idx, test_idx) in enumerate(gene_cv.split(X, y, groups), start=1):
    model = make_model(frozen_params)
    model.fit(X[train_idx], y[train_idx])
    fold_pred = model.predict(X[test_idx])

    oof_true.append(y[test_idx])
    oof_pred.append(fold_pred)

    fold_frame = analysis_df.iloc[test_idx].reset_index().rename(columns={"index": "source_index"}).copy()
    fold_frame["row_index"] = test_idx
    fold_frame["fold_id"] = fold_id
    fold_frame["group"] = groups[test_idx]
    fold_frame["y_true"] = y[test_idx]
    fold_frame["y_pred"] = fold_pred
    fold_frame["residual"] = fold_frame["y_true"] - fold_frame["y_pred"]
    fold_frame["abs_error"] = fold_frame["residual"].abs()
    oof_frames.append(fold_frame)

    fold_rows.append({
        "fold_id": fold_id,
        "n_train": int(len(train_idx)),
        "n_test": int(len(test_idx)),
        "n_train_groups": int(len(np.unique(groups[train_idx]))),
        "n_test_groups": int(len(np.unique(groups[test_idx]))),
    })

predictions_df = pd.concat(oof_frames, ignore_index=True)
metrics_df = pd.DataFrame([evaluate(np.concatenate(oof_true), np.concatenate(oof_pred))])
fold_summary = pd.DataFrame(fold_rows)
comparison = pd.concat([baseline_metrics.assign(run="baseline_by_gene_mrna"), metrics_df.assign(run="ablation")], ignore_index=True)
delta_vs_baseline = metrics_df.iloc[0] - baseline_metrics.iloc[0]
delta_df = pd.DataFrame({"metric": delta_vs_baseline.index, "delta_vs_baseline": delta_vs_baseline.values})
fold_summary


,fold_id,n_train,n_test,n_train_groups,n_test_groups
0,1,23692,11107,54,16
1,2,23698,11101,54,15
2,3,23722,11077,54,17


## Fold Summary

## Baseline Comparison

In [7]:
comparison

,spearman,pearson,rmse,mae,run
0,0.458860,0.453422,31.421689,24.568816,baseline_by_gene_mrna
1,0.483176,0.479460,30.964610,24.114665,ablation


## Metric Delta Vs Baseline

In [8]:
delta_df

,metric,delta_vs_baseline
0,spearman,0.024316
1,pearson,0.026038
2,rmse,-0.457079
3,mae,-0.454151


## Overall Metrics

In [9]:
metrics_df

,spearman,pearson,rmse,mae
0,0.483176,0.47946,30.96461,24.114665


## Concentration Ranking Slice

These tables show where XGBoost preserves ranking poorly or well across concentration alone and the key concentration combinations after the ablation.

In [10]:
spearman_by_concentration = grouped_spearman(predictions_df, ["Concentration_nM"], min_samples=20)
spearman_by_concentration_gene = grouped_spearman(predictions_df, ["Concentration_nM", "group"], min_samples=10)
spearman_by_concentration_patent = grouped_spearman(predictions_df, ["Concentration_nM", "patent_group"], min_samples=10)

spearman_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,spearman,mae,mean_true,mean_pred
0,0.00064,113,-0.101566,13.296286,-1.467611,6.693690
1,6.66670,49,-0.054375,15.440500,78.425510,73.592354
2,0.01000,197,-0.008004,26.161063,36.815228,18.351851
3,0.00300,64,0.007166,17.747605,3.631875,-4.561400
4,0.00320,113,0.065902,16.262903,1.580354,11.332241
5,0.00100,66,0.066431,20.283872,7.866212,-6.593073
6,150.00000,86,0.097794,27.654246,36.667326,59.897087
7,0.20000,118,0.100443,37.454375,17.501441,13.286599
8,2.22220,48,0.109660,13.868307,78.291042,73.358482
9,0.05000,292,0.129340,18.025597,16.818836,25.536474


## Concentration + Gene Hotspots

In [11]:
spearman_by_concentration_gene.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,group,n_samples,spearman,mae,mean_true,mean_pred
0,5.00000,HSD17B13,52,-0.521045,45.786571,34.843462,64.679207
1,5.00000,AGT,26,-0.418875,26.597832,70.858077,45.389626
2,0.01600,HSD17B13,13,-0.416667,22.549500,38.220000,15.670501
3,0.00320,HSD17B13,13,-0.394444,21.014080,12.342308,2.852548
4,2.22200,HSD17B13,16,-0.217647,22.263109,87.950000,70.315735
5,0.24700,HSD17B13,16,-0.201619,31.596311,78.275000,53.815002
6,2.22220,AGT,20,-0.168866,13.700581,82.997500,74.083038
7,0.00100,AGT,20,-0.166290,13.393230,-0.747500,1.577615
8,20.00000,LPA,36,-0.165251,16.196135,72.276944,77.315331
9,10.00000,LPA,57,-0.160625,26.040742,67.965439,79.730934


## Concentration + Patent Hotspots

In [12]:
spearman_by_concentration_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,patent_group,n_samples,spearman,mae,mean_true,mean_pred
0,5.00000,CN117070517A,26,-0.418875,26.597832,70.858077,45.389626
1,0.10000,CN112313335A,182,-0.409576,85.325619,98.655549,13.329930
2,10.00000,WO2023091644A2,117,-0.385996,43.424264,58.020085,38.434303
3,10.00000,CN112313335A,181,-0.351212,53.800777,92.925635,39.124859
4,1.00000,CN117448322A,20,-0.284424,49.093693,47.430000,28.960169
5,2.22200,CN116940681A,16,-0.217647,22.263109,87.950000,70.315735
6,0.24700,CN116940681A,16,-0.201619,31.596311,78.275000,53.815002
7,0.01000,CN117448322A,45,-0.179464,33.862166,34.295556,14.214050
8,0.01000,CN117327698A,51,-0.169156,14.795740,26.669804,20.768744
9,2.22220,WO2023088427A1,20,-0.168866,13.700581,82.997500,74.083038


## Experimental Feature Issues

In [13]:
issue_by_cell_type = issue_summary(predictions_df, "Cell_Type", min_samples=30)
issue_by_concentration = issue_summary(predictions_df, "Concentration_nM", min_samples=20)
issue_by_time = issue_summary(predictions_df, "Time_of_administration_h", min_samples=20)
issue_by_patent = issue_summary(predictions_df, "patent_group", min_samples=20)
issue_by_authorization = issue_summary(predictions_df, "Authorization_status", min_samples=20)

issue_by_cell_type.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Cell_Type,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Non-human hepatocytes,94,27.392595,34.367683,-0.077386,17.238298,43.864532
1,Primary mouse hepatocytes,307,49.628125,62.404964,0.092840,24.124267,47.125240
2,Primary human hepatocytes,2837,28.209263,34.217064,0.116763,51.122904,45.548344
3,Primary Cynomolgus Monkey Hepatocytes,4146,25.176619,31.914294,0.322139,33.559431,37.277397
4,Hep3B,11794,27.876196,35.064706,0.392598,42.257230,37.963730
5,HepG2,1528,21.135292,27.843137,0.424064,39.763298,46.189976
6,Huh7,1546,22.788611,28.844719,0.551252,40.624314,38.294716
7,Human iPSC-derived cortical neurons,199,22.550438,26.270815,0.565908,42.732663,40.851738
8,Hela,1643,22.030795,27.259753,0.583596,39.570590,40.093300
9,H1299 Cells,1464,12.318865,15.577694,0.590612,68.438388,68.254257


## Concentration Issue Summary

In [14]:
issue_by_concentration.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Concentration_nM,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,0.00064,113,13.296286,16.588470,-0.101566,-1.467611,6.693690
1,6.66670,49,15.440500,17.983779,-0.054375,78.425510,73.592354
2,0.01000,197,26.161063,32.587726,-0.008004,36.815228,18.351851
3,0.00300,64,17.747605,20.975318,0.007166,3.631875,-4.561400
4,0.00320,113,16.262903,19.717231,0.065902,1.580354,11.332241
5,0.00100,66,20.283872,24.102647,0.066431,7.866212,-6.593073
6,150.00000,86,27.654246,35.569068,0.097794,36.667326,59.897087
7,0.20000,118,37.454375,43.108764,0.100443,17.501441,13.286599
8,2.22220,48,13.868307,16.555050,0.109660,78.291042,73.358482
9,0.05000,292,18.025597,23.218361,0.129340,16.818836,25.536474


## Time Issue Summary

In [15]:
issue_by_time.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Time_of_administration_h,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,40.0,194,15.710119,21.292907,0.230669,27.380412,28.262068
1,24.0,23628,25.882174,33.016862,0.432163,41.515870,41.042221
2,72.0,1387,17.931039,22.269440,0.495744,15.584420,25.309313
3,48.0,7861,20.152015,25.806741,0.560056,48.546870,44.585342
4,168.0,199,22.550438,26.270815,0.565908,42.732663,40.851738


## Patent Issue Summary

In [16]:
issue_by_patent.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,patent_group,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,CN112313335A,363,69.606621,72.150045,-0.758876,95.798485,26.191866
1,CN117448322A,131,44.999170,51.458752,-0.120558,45.182443,15.073058
2,WO2023091644A2,1434,31.633085,37.510336,-0.085638,48.256402,40.649914
3,WO2022121959A1,97,31.616203,43.240444,-0.084975,64.074742,68.986320
4,TW202321444A,94,27.392595,34.367683,-0.077386,17.238298,43.864532
5,CN117210468A,222,19.596031,26.035764,0.002196,25.251351,31.889175
6,CN116801886A,262,28.315013,37.464359,0.066637,50.140802,56.453423
7,CN116732034A,30,40.078115,42.303873,0.131034,83.838000,43.759884
8,WO2023056446A1,223,21.527625,27.399083,0.144442,61.098655,55.072372
9,CN117106781A,20,23.499813,28.052175,0.145865,65.312500,41.949688


## Authorization Issue Summary

In [17]:
issue_by_authorization.sort_values(["spearman", "mae"], ascending=[True, False]).head(20)

,Authorization_status,n_samples,mae,rmse,spearman,mean_true,mean_pred
0,Published,222,19.596031,26.035764,0.002196,25.251351,31.889175
1,Substantive Examination,15576,27.071913,33.799264,0.484743,46.018578,34.863930
2,Unknown Status,12453,23.012663,30.085887,0.530909,33.370768,44.557716
3,Withdrawn,1078,22.827416,28.238645,0.545178,30.062894,36.410595
4,UNKNOWN,2333,13.263848,16.814567,0.630900,63.828521,64.876839
5,Granted,1623,21.259998,26.944928,0.639437,48.829341,45.598202


## Prediction Preview

In [18]:
predictions_df[["group", "Cell_Type", "Concentration_nM", "Time_of_administration_h", "patent_group", "y_true", "y_pred", "residual", "abs_error"]].head(20)

,group,Cell_Type,Concentration_nM,Time_of_administration_h,patent_group,y_true,y_pred,residual,abs_error
0,CTNNB1,Hela,100.0,48.0,CN107365771B,88.00,83.585770,4.414230,4.414230
1,CTNNB1,Hela,100.0,48.0,CN107365771B,90.00,101.239922,-11.239922,11.239922
2,CTNNB1,Hela,100.0,48.0,CN107365771B,90.00,70.778458,19.221542,19.221542
3,CTNNB1,Hela,100.0,48.0,CN107365771B,89.00,100.184090,-11.184090,11.184090
4,CTNNB1,Hela,100.0,48.0,CN107365771B,87.00,84.466148,2.533852,2.533852
5,CTNNB1,Hela,100.0,48.0,CN107365771B,91.00,114.466476,-23.466476,23.466476
6,CTNNB1,Hela,100.0,48.0,CN107365771B,62.00,60.465870,1.534130,1.534130
7,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,93.62,100.159019,-6.539019,6.539019
8,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,92.55,88.254944,4.295056,4.295056
9,CTNNB1,Hep3B,10.0,24.0,WO2023003995A8,93.48,124.382126,-30.902126,30.902126


## Interpretation Notes

- Removing the three target-specific hotspots `APP @ 20 nM`, `INHBE @ 100 nM`, and `PCSK9 @ 100 nM` gives a clearly stronger improvement than the patent-hotspot ablation.
- Overall metrics improve from `Spearman 0.4589` to `0.4832`, `Pearson 0.4534` to `0.4795`, `RMSE 31.42` to `30.96`, and `MAE 24.57` to `24.11`.
- This means those target-specific transfer hotspots were materially harming the model, and the improvement is large enough to matter.
- Even after this removal, the weakest `Concentration + Gene` rows are still dominated by `HSD17B13` and some `AGT`/`LPA` combinations. That suggests the by-gene XGBoost failure is still partly concentrated in specific targets rather than being driven only by global assay conditions.
- On the patent side, the same broad source warnings remain: `CN112313335A`, `WO2023091644A2`, `WO2022121959A1`, `TW202321444A`, and `US20220117999A1` still show weak ranking or high error. So this ablation improves target-specific transfer, but it does not remove the patent/source heterogeneity problem.
- Cell-type behavior also improves only partially. `Primary mouse hepatocytes` is still a serious warning sign, although it is less catastrophic than in the original by-gene notebook. `Non-human hepatocytes` also remains weak.
- Time-of-administration again does not look like the main issue. This notebook still does not support broad deletion of all `72h` rows.
- Overall, this notebook points more strongly toward target-specific hotspot filtering than the patent-hotspot notebook did.

## Conclusions And Next Step

- This is the stronger of the two XGBoost ablations so far. If we are choosing one axis to prioritize first, the target-specific hotspot ablation looks more promising than removing the single patent hotspot.
- `APP @ 20 nM`, `INHBE @ 100 nM`, and `PCSK9 @ 100 nM` should stay on the high-priority candidate-removal list for pre-deep-learning cleanup.
- But the remaining tables also say we are not done: `HSD17B13` now looks like the most concentrated unresolved target-specific problem, while `CN112313335A` and `US20220117999A1` remain strong patent-level warnings.
- The best next step is to build one overlap shortlist combining:
  - RF suspicious hotspots,
  - original XGBoost by-gene suspicious hotspots,
  - and the remaining worst rows after this ablation.
- Then we should test one final focused ablation that removes only the shared highest-confidence hotspots rather than broad groups.
